# Frontier Intelligence Dynamics

This notebook combines model release metadata, benchmark scores, OpenRouter pricing availability, and Hugging Face enrichment into one registry for frontier analysis.

Data source note:
- Benchmark coverage comes from the ZeroEval API snapshot tracked in this repo.
- Model coverage is limited to what ZeroEval currently exposes, so a missing latest model may reflect source coverage rather than notebook logic.
- This notebook reads cached local datasets by default for speed; CI is expected to keep them fresh on a weekly cadence.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

BASE_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
SRC_DIR = BASE_DIR / "src"
for path in (BASE_DIR, SRC_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"Base dir: {BASE_DIR}")


In [ ]:
from research_data.api import build_frontier_model_registry, frontier_summary, monthly_model_releases

frontier_registry = build_frontier_model_registry(base_dir=BASE_DIR, refresh=False)
summary = frontier_summary(base_dir=BASE_DIR, refresh=False)

benchmark_path = BASE_DIR / 'data' / 'normalized' / 'llm_benchmarks' / 'llm_benchmarks.parquet'
if benchmark_path.exists():
    benchmark_frame = pd.read_parquet(benchmark_path)
else:
    benchmark_frame = pd.read_csv(benchmark_path.with_suffix('.csv'))

latest_benchmark_scraped_at = pd.to_datetime(benchmark_frame['scraped_at'], errors='coerce', utc=True).max()
latest_frontier_release_date = pd.to_datetime(frontier_registry['release_date'], errors='coerce').max()

freshness_summary = pd.DataFrame(
    [
        {'metric': 'latest_zeroeval_scraped_at', 'value': latest_benchmark_scraped_at},
        {'metric': 'latest_frontier_release_date', 'value': latest_frontier_release_date},
        {'metric': 'frontier_registry_rows', 'value': len(frontier_registry)},
    ]
)

print(summary)
freshness_summary


## Monthly Release Counts


In [ ]:
monthly_releases = monthly_model_releases(base_dir=BASE_DIR, refresh=False)
print(monthly_releases.tail(12).to_string(index=False))
monthly_releases.tail(12)


## Frontier-Candidate Slice


In [ ]:
frontier_candidates = frontier_registry[frontier_registry["is_large_model"].fillna(False)].copy()
frontier_candidates = frontier_candidates.sort_values("frontier_score", ascending=False)
frontier_candidates[[
    "model_id",
    "organization",
    "release_date",
    "context_window",
    "gpqa",
    "swe_bench",
    "frontier_score",
    "is_on_openrouter",
]].head(20)


In [ ]:
organization_summary = (
    frontier_registry.groupby("organization", as_index=False)
    .agg(
        model_count=("model_id", "count"),
        large_model_count=("is_large_model", "sum"),
        avg_gpqa=("gpqa", "mean"),
        avg_swe_bench=("swe_bench", "mean"),
        avg_frontier_score=("frontier_score", "mean"),
    )
    .sort_values(["large_model_count", "avg_frontier_score"], ascending=[False, False])
)
organization_summary.head(15)
